# Analyzing Job Skills with LAiSER

This notebook helps job seekers understand the most in-demand skills for their target roles by extracting and visualizing skills from job descriptions.

- Author: Satya Phanindra Kumar Kalaga
- Date: September 2025

## 1. Setup

First, let's install the `laiser` package and other necessary libraries.

In [ ]:
!pip install laiser pandas matplotlib wordcloud

## 2. Import Libraries

In [ ]:
import pandas as pd
from laiser.skill_extractor_refactored import SkillExtractorRefactored
import matplotlib.pyplot as plt
from wordcloud import WordCloud

## 3. Initialize the Skill Extractor

We'll initialize the `SkillExtractorRefactored` from the `laiser` package. You will need to provide your Hugging Face model ID and token. For this example, we will use a CPU-only environment.

In [ ]:
# Replace with your Hugging Face model ID and token
model_id = "your_model_id"  # e.g., "mistralai/Mistral-7B-Instruct-v0.1"
hf_token = "your_hf_token"

try:
    se = SkillExtractorRefactored(model_id=model_id, hf_token=hf_token, use_gpu=False)
    print("Skill Extractor initialized successfully!")
except Exception as e:
    print(f"Error initializing Skill Extractor: {e}")

## 4. Load Job Postings Data

We'll load a sample dataset of job postings from a CSV file. This dataset contains job descriptions that we will analyze.

In [ ]:
url = 'https://raw.githubusercontent.com/LAiSER-Software/datasets/refs/heads/master/jobs-data/linkedin_jobs_sample_36rows.csv'
job_postings = pd.read_csv(url)
job_postings = job_postings[['description', 'job_id']].dropna()
print("Job postings data loaded successfully:")
job_postings.head()

## 5. Extract Skills from Job Descriptions

Now, we'll use the `extract_and_align` method from the `laiser` skill extractor to identify skills in the job descriptions.

In [ ]:
try:
    extracted_skills = se.extract_and_align(
        job_postings,
        id_column="job_id",
        text_columns=["description"],
        input_type='job_desc'
    )
    print("Skills extracted successfully!")
    print(extracted_skills.head())
except Exception as e:
    print(f"Error during skill extraction: {e}")

## 6. Visualize the Extracted Skills

Let's visualize the most common skills found in the job postings. A bar chart and a word cloud are great ways to see which skills are most in-demand.

In [ ]:
if 'extracted_skills' in locals() and not extracted_skills.empty:
    skill_counts = extracted_skills['skill'].value_counts().nlargest(20)

    # Bar Chart
    plt.figure(figsize=(12, 8))
    skill_counts.sort_values().plot(kind='barh')
    plt.title('Top 20 Most In-Demand Skills')
    plt.xlabel('Frequency')
    plt.ylabel('Skill')
    plt.show()

    # Word Cloud
    all_skills = ' '.join(extracted_skills['skill'].dropna())
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_skills)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Extracted Skills')
    plt.show()
else:
    print("No skills were extracted to visualize.")